In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Ensure backend root is in sys.path
notebook_dir = Path(os.getcwd())
backend_dir = notebook_dir.parents[3] if "agents" in str(notebook_dir) else notebook_dir
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

# Load GROQ_API_KEY from backend/.env
dotenv_path = backend_dir / ".env"
load_dotenv(dotenv_path)

if not os.environ.get("GROQ_API_KEY"):
    print("Warning: GROQ_API_KEY not found in backend/.env")
else:
    print("GROQ_API_KEY loaded successfully.")

GROQ_API_KEY loaded successfully.


In [2]:
from src.agents.knowledge_grap_manager_llm.state import (
    WikiState,
    DBSource,
    ExtractedProduct,
    ExtractedReview,
    ReviewSynthesis,
    WikiPage,
    Contradiction,
    LogEntry,
    ValidationResult,
)
from src.agents.knowledge_grap_manager_llm.graph import build_wiki_graph
from src.agents.knowledge_grap_manager_llm import run_wiki_agent, query_wiki

In [3]:
test_data_dir = backend_dir / "test_data"

collected_data = []

products_csv = test_data_dir / "products.csv"
if products_csv.exists():
    collected_data.append({
        "category": "catalog",
        "data": products_csv.read_text(encoding="utf-8"),
        "format": "csv",
    })

reviews_csv = test_data_dir / "reviews.csv"
if reviews_csv.exists():
    collected_data.append({
        "category": "review",
        "data": reviews_csv.read_text(encoding="utf-8"),
        "format": "csv",
    })

promotions_csv = test_data_dir / "promotions.csv"
if promotions_csv.exists():
    collected_data.append({
        "category": "promotion",
        "data": promotions_csv.read_text(encoding="utf-8"),
        "format": "csv",
    })

print(f"Loaded {len(collected_data)} dataset items:")
for item in collected_data:
    print(f" - {item['category']} ({len(item['data'].splitlines())} lines)")

Loaded 3 dataset items:
 - catalog (10 lines)
 - review (28 lines)
 - promotion (6 lines)


In [4]:
from src.agents.knowledge_grap_manager_llm.nodes.collect_data import collect_data

test_state: WikiState = {
    "merchant_id": "merchant_test_01",
    "wiki_section": "knowledge",
    "wiki_base_path": str(backend_dir / "merchant_knowledge_test"),
    "collected_data": collected_data,
    "classified_sources": {},
    "extracted_products": [],
    "extracted_reviews": [],
    "review_syntheses": [],
    "entities": [],
    "existing_pages": {},
    "pages_to_create": [],
    "pages_to_update": [],
    "contradictions": [],
    "generated_pages": [],
    "index_updates": [],
    "log_entry": {
        "timestamp": "",
        "operation": "ingest",
        "section": "knowledge",
        "files_processed": [],
        "products_added": 0,
        "products_updated": 0,
        "reviews_processed": 0,
        "pages_created": [],
        "pages_updated": [],
        "conflicts": 0,
        "status": "PENDING",
        "errors": [],
    },
    "validation_result": {
        "orphan_pages": [],
        "duplicate_products": [],
        "missing_reviews": [],
        "conflicting_specs": [],
        "outdated_prices": [],
        "broken_links": [],
        "health_score": 1.0,
    },
    "validation_errors": [],
    "status": "running",
    "error": "",
}

collect_res = collect_data(test_state)
test_state.update(collect_res)
print("Classified categories:", list(test_state["classified_sources"].keys()))

 Collected DB data -> Categories: ['catalog', 'review', 'promotion']
Classified categories: ['catalog', 'review', 'promotion']


In [5]:
from src.agents.knowledge_grap_manager_llm.nodes.extract_entities import extract_entities

entities_res = extract_entities(test_state)
test_state.update(entities_res)
print(f"Extracted {len(test_state['extracted_products'])} products:")
for p in test_state["extracted_products"][:]:
    print(f" - {p['name']} ({p['slug']}): {p['price']} {p['currency']}")

  Extracted 9 unique product entities
Extracted 9 products:
 - iPhone 15 (iphone-15): 79900 INR
 - iPhone 15 Pro (iphone-15-pro): 134900 INR
 - Samsung Galaxy S24 (samsung-galaxy-s24): 74999 INR
 - Samsung Galaxy S24 Ultra (samsung-galaxy-s24-ultra): 129999 INR
 - OnePlus 12 (oneplus-12): 64999 INR
 - Sony WH-1000XM5 (sony-wh-1000xm5): 29990 INR
 - Apple AirPods Pro 2 (apple-airpods-pro-2): 24900 INR
 - iPad Air M2 (ipad-air-m2): 59900 INR
 - Samsung Galaxy Tab S9 (samsung-galaxy-tab-s9): 74999 INR


In [6]:
from src.agents.knowledge_grap_manager_llm.nodes.extract_reviews import extract_reviews

reviews_res = extract_reviews(test_state)
test_state.update(reviews_res)
print(f"Processed {len(test_state['extracted_reviews'])} reviews into {len(test_state['review_syntheses'])} product syntheses:")
for s in test_state["review_syntheses"][:3]:
    print(f" - {s['product_name']}: {s['avg_rating']}/5 ({s['total_reviews']} reviews) -> {s['sentiment_summary'][:80]}...")

  Processed 27 reviews for 9 products
Processed 27 reviews into 9 product syntheses:
 - iPhone 15: 4.2/5 (5 reviews) -> Overall, customers rate the iPhone 15 positively with an average of 4.2/5, highl...
 - iPhone 15 Pro: 4.7/5 (3 reviews) -> Customers are highly satisfied with the iPhone 15 Pro, praising its premium tita...
 - Samsung Galaxy S24: 4.0/5 (4 reviews) -> Overall customers view the Samsung Galaxy S24 positively, giving it an average 4...


In [8]:
for s in test_state["review_syntheses"][:]:
    print(f" - {s['product_name']}: {s['avg_rating']}/5 ({s['total_reviews']} reviews) -> {s['sentiment_summary'][:]}...")

 - iPhone 15: 4.2/5 (5 reviews) -> Overall, customers rate the iPhone 15 positively with an average of 4.2/5, highlighting its camera quality and smooth performance, though some note the high price and lack of revolutionary features. 📄 user_input...
 - iPhone 15 Pro: 4.7/5 (3 reviews) -> Customers are highly satisfied with the iPhone 15 Pro, praising its premium titanium build, outstanding camera quality, and fast performance, yielding an average rating of 4.7 out of 5....
 - Samsung Galaxy S24: 4.0/5 (4 reviews) -> Overall customers view the Samsung Galaxy S24 positively, giving it an average 4.0/5 rating. Reviewers praise its AI capabilities, display quality, and flagship value, while a few feel it falls short of expectations for a premium device....
 - Samsung Galaxy S24 Ultra: 4.7/5 (3 reviews) -> Customers are overwhelmingly positive, highlighting the phone’s exceptional photography and suitability as a business device, though the price is seen as a drawback....
 - OnePlus 12: 4.3

In [9]:
from src.agents.knowledge_grap_manager_llm.nodes.search_existing_wiki import search_existing_wiki
from src.agents.knowledge_grap_manager_llm.nodes.knowledge_diff import knowledge_diff

search_res = search_existing_wiki(test_state)
test_state.update(search_res)

diff_res = knowledge_diff(test_state)
test_state.update(diff_res)
print(f"Diff results: {len(test_state['pages_to_create'])} pages to create, {len(test_state['pages_to_update'])} to update, {len(test_state['contradictions'])} contradictions")

  Wiki search: 9 existing pages found, 0 new products to create
  Knowledge diff complete: 0 to create, 1 to update, 0 conflicts
Diff results: 0 pages to create, 1 to update, 0 contradictions


In [12]:
test_state.get("pages_to_update")

[{'slug': 'samsung-galaxy-tab-s9',
  'title': 'Samsung Galaxy Tab S9',
  'page_type': 'products',
  'section': 'knowledge',
  'file_path': 'c:\\Users\\ps302\\OneDrive\\Desktop\\Razorpay\\backend\\merchant_knowledge_test\\wiki\\knowledge\\products\\samsung-galaxy-tab-s9.md',
  'content': '',
  'sources': ['db_catalog_1'],
  'links': []}]

In [13]:
from src.agents.knowledge_grap_manager_llm.nodes.create_pages import create_pages

create_res = create_pages(test_state)
test_state.update(create_res)
print(f"Generated {len(test_state['generated_pages'])} wiki pages.")

Generated 0 wiki pages.


In [14]:
from src.agents.knowledge_grap_manager_llm.nodes.cross_reference import cross_reference
from src.agents.knowledge_grap_manager_llm.nodes.update_index import update_index
from src.agents.knowledge_grap_manager_llm.nodes.append_log import append_log
from src.agents.knowledge_grap_manager_llm.nodes.validate_wiki import validate_wiki

test_state.update(cross_reference(test_state))
test_state.update(update_index(test_state))
test_state.update(append_log(test_state))
val_res = validate_wiki(test_state)
test_state.update(val_res)

print("Validation Health Score:", test_state["validation_result"]["health_score"])

  Index updated: 12 pages indexed in 'knowledge'
  Log appended: status=SUCCESS

  ---- Wiki Health Report ----
  Orphan pages: 0
  Duplicate products: 0
  Conflicting specs: 0
  Missing reviews: 0
  Broken links: 0
  Health score: 100%
  ----------------------------

Validation Health Score: 1.0


In [ ]:
wiki_output = str(backend_dir / "merchant_knowledge_test")

print("--- Running Knowledge Base Pipeline ---")
knowledge_state = run_wiki_agent(
    merchant_id="merchant_electronics_01",
    wiki_base_path=wiki_output,
    wiki_section="knowledge",
    collected_data=collected_data,
)

print("\n--- Running Marketing Intelligence Pipeline ---")
marketing_state = run_wiki_agent(
    merchant_id="merchant_electronics_01",
    wiki_base_path=wiki_output,
    wiki_section="marketing",
    collected_data=collected_data,
)

In [15]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

notebook_dir = Path(os.getcwd())
backend_dir = notebook_dir.parents[3] if "agents" in str(notebook_dir) else notebook_dir
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

load_dotenv(backend_dir / ".env")

from src.agents.knowledge_grap_manager_llm import query_wiki

wiki_output = str(backend_dir / "merchant_knowledge_test")

query_result = query_wiki(
    query="What is the price, battery life, and overall customer sentiment for iPhone 15 Pro?",
    wiki_base_path=wiki_output,
    merchant_id="merchant_electronics_01",
)

print("--- Knowledge Graph Query Response ---")
print(query_result["answer"])
print("\nSources cited:", query_result["sources"])

--- Knowledge Graph Query Response ---
**iPhone 15 Pro**

| Item | Detail |
|------|--------|
| **Price** | ₹134,900 INR【knowledge/products/iphone-15-pro.md】 |
| **Battery life** | Up to 23 hours of video playback【knowledge/products/iphone-15-pro.md】 |
| **Overall customer sentiment** | 4.7 / 5 rating from 3 reviews – customers praise the premium titanium build, outstanding camera, and fast performance, with a minor complaint about warmth under heavy load【knowledge/products/iphone-15-pro.md】 |

*Key take‑away:* The iPhone 15 Pro commands a premium price but enjoys strong customer approval, especially for its build quality and camera performance.

Sources cited: ['knowledge/products/iphone-15.md', 'knowledge/products/iphone-15-pro.md', 'marketing/promotions/iphone-15-pro.md', 'marketing/popular/iphone-15.md', 'marketing/index.md']


In [16]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

notebook_dir = Path(os.getcwd())
backend_dir = notebook_dir.parents[3] if "agents" in str(notebook_dir) else notebook_dir
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

load_dotenv(backend_dir / ".env")

from src.agents.knowledge_grap_manager_llm import query_wiki

wiki_output = str(backend_dir / "merchant_knowledge_test")

query_result = query_wiki(
    query="What is the price, battery life, and overall customer sentiment for iPhone 17 Pro?",
    wiki_base_path=wiki_output,
    merchant_id="merchant_electronics_01",
)

print("--- Knowledge Graph Query Response ---")
print(query_result["answer"])
print("\nSources cited:", query_result["sources"])

--- Knowledge Graph Query Response ---
I’m sorry, but the store knowledge base does not contain any information about an **iPhone 17 Pro**. The available product pages cover the iPhone 15 and iPhone 15 Pro, but there is no entry for a 17 Pro model, so I can’t provide its price, battery life, or customer sentiment.

Sources cited: ['knowledge/products/iphone-15-pro.md', 'marketing/promotions/iphone-15-pro.md', 'knowledge/products/iphone-15.md', 'marketing/popular/iphone-15.md', 'marketing/index.md']


In [17]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

notebook_dir = Path(os.getcwd())
backend_dir = notebook_dir.parents[3] if "agents" in str(notebook_dir) else notebook_dir
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

load_dotenv(backend_dir / ".env")

from src.agents.knowledge_grap_manager_llm import query_wiki

wiki_output = str(backend_dir / "merchant_knowledge_test")

query_result = query_wiki(
    query="hey, i need to buy some choclates can you help me out",
    wiki_base_path=wiki_output,
    merchant_id="merchant_electronics_01",
)

print("--- Knowledge Graph Query Response ---")
print(query_result["answer"])
print("\nSources cited:", query_result["sources"])

--- Knowledge Graph Query Response ---
I’m sorry, but the store knowledge base for **merchant_electronics_01** does not contain any information about chocolates or confectionery items. All available product data is limited to electronics such as smartphones, headphones, and tablets. If you’re looking for chocolates, you might want to check a different store or category that specializes in food or sweets.

Sources cited: ['marketing/index.md', 'index.md', 'knowledge/index.md', 'knowledge/products/iphone-15.md', 'marketing/popular/iphone-15.md']
